<a href="https://colab.research.google.com/github/hinamehboobcs/Industry-Project/blob/main/PPO_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install stable-baselines3 gymnasium
!pip install -U gymnasium stable-baselines3
import gymnasium as gym

In [16]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure project path is visible
sys.path.append("/content/Project")

print("Setup complete")

Setup complete


In [17]:
os.makedirs("/content/Project/results", exist_ok=True)
os.makedirs("/content/Project/ppo", exist_ok=True)

In [18]:
visits = pd.read_csv("/content/Project/data/visits_data.csv")
carers = pd.read_csv("/content/Project/data/carers_data.csv")

print(visits.shape)
print(carers.shape)

(657, 9)
(263, 12)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [23]:
from PPO.preprocessing import preprocess_visits, preprocess_carers

visits = preprocess_visits(visits)
carers = preprocess_carers(carers)

print(visits.shape)
print(carers.shape)

(657, 12)
(263, 15)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [24]:
print(visits.columns)
print(carers.columns)

Index(['VisitID', 'PatientID', 'Day', 'Latitude', 'Longitude',
       'VisitDurationMinutes', 'TimeWindowStart', 'TimeWindowEnd',
       'PriorityLevel', 'TimeWindowStartMin', 'TimeWindowEndMin',
       'DayEncoded'],
      dtype='object')
Index(['CarerID', 'Latitude', 'Longitude', 'MaxTravelDistanceMiles',
       'WorkingDays', 'OffDays', 'ShiftStart', 'ShiftEnd',
       'ContractHoursPerWeek', 'MaxDailyHours', 'MaxWeeklyHours', 'Status',
       'WorkingDaysSet', 'ShiftStartMin', 'ShiftEndMin'],
      dtype='object')


In [28]:
from PPO.distance_engine import build_distance_matrix

distance_matrix = build_distance_matrix(visits, carers)

print(distance_matrix.shape)
print(distance_matrix[:2, :2])

[DISTANCE] Building matrix...
[DISTANCE] Done: (657, 263)
(657, 263)
[[ 3.72097831  3.72097831]
 [25.158148   25.158148  ]]


In [29]:
from PPO.train import train_ppo

model, env = train_ppo(visits, carers, distance_matrix)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------
| time/              |     |
|    fps             | 59  |
|    iterations      | 1   |
|    time_elapsed    | 4   |
|    total_timesteps | 256 |
----------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------------
| time/                   |              |
|    fps                  | 58           |
|    iterations           | 2            |
|    time_elapsed         | 8            |
|    total_timesteps      | 512          |
| train/                  |              |
|    approx_kl            | 3.058929e-05 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -5.57        |
|    explained_variance   | 0.00112      |
|    learning_rate        | 0.0003       |
|    loss                 | 2.36e+05     |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.00298     |
|    value_loss           | 4.58e+05     |
------------------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 657           |
|    ep_rew_mean          | -2.77e+04     |
| time/                   |               |
|    f

In [32]:
# Fix IndentationError in PPO/metrics.py
# The error was: IndentationError: expected an indented block after function definition on line 19
# This block rewrites the file with the corrected indentation and improved robustness.
file_content_metrics = """import pandas as pd
import numpy as np

def evaluate_system(assignments_df, visits, carers):
    if not isinstance(assignments_df, pd.DataFrame) or assignments_df.empty:
        print("Warning: assignments_df is empty or not a DataFrame. Returning empty metrics.")
        # Return an empty DataFrame with the expected columns to avoid downstream errors
        return pd.DataFrame({'Metric': [], 'Value': []})

    # Placeholder logic for evaluation
    # This is a simplified example; a real evaluation would involve more complex calculations
    total_visits = len(visits)
    total_carers = len(carers)
    assigned_visits = len(assignments_df)

    # Example: Calculate a dummy utilization metric (assuming assignments_df has 'CarerID')
    carers_assigned_visits = assignments_df['CarerID'].nunique() if 'CarerID' in assignments_df.columns else 0
    carer_utilization_rate = carers_assigned_visits / total_carers if total_carers > 0 else 0

    metrics_data = {
        'Total Visits': [total_visits],
        'Total Carers': [total_carers],
        'Assigned Visits': [assigned_visits],
        'Carer Utilization Rate': [carer_utilization_rate]
    }
    return pd.DataFrame(metrics_data)

def save_metrics(metrics_df, path):
    metrics_df.to_csv(path, index=False)
    print(f"Metrics saved to {path}")
"""

with open("/content/Project/PPO/metrics.py", "w") as f:
    f.write(file_content_metrics)

# Now, import and use the corrected functions
from PPO.metrics import evaluate_system, save_metrics
import pandas as pd # Ensure pandas is imported for DataFrame creation

# Note: 'assignments' variable is not yet defined. This will likely cause a NameError next.
# You would typically get assignments by running the trained model through the environment.
# For now, we will assume 'assignments' needs to be generated or passed from a previous step.
# As a temporary measure to proceed past the IndentationError, I will define a dummy assignments_df.
assignments = pd.DataFrame(columns=['VisitID', 'CarerID', 'StartTime', 'EndTime'])

metrics_df = evaluate_system(assignments, visits, carers)

print(metrics_df)

# Fix: Use the absolute path for saving the metrics file
save_metrics(metrics_df, "/content/Project/results/ppo_metrics.csv")

Empty DataFrame
Columns: [Metric, Value]
Index: []
Metrics saved to /content/Project/results/ppo_metrics.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [40]:
# Generate assignments using the trained model
# This involves running the model in the environment to collect actions (assignments).

# Reset the environment to get an initial observation
# Fix: env.reset() for DummyVecEnv returns only observations, not (obs, info) tuple.
obs_batch = env.reset()
current_obs = obs_batch[0] # Extract the single dictionary observation from the batch

# Initialize lists to store assignment details
all_actions = []
all_observations = [] # Store full observations including 'visit_id' for later use

# Run the model for a number of steps (e.g., for each visit)
num_steps = visits.shape[0]

for _ in range(num_steps):
    # Store the full current observation before processing for the model
    all_observations.append(current_obs)

    # Filter out 'visit_id' from the observation passed to the model,
    # as it's likely an identifier and not a feature for the policy network.
    # The model's observation space probably doesn't include a space for 'visit_id'.
    model_input_obs = {k: v for k, v in current_obs.items() if k != 'visit_id'}

    action, _states = model.predict(model_input_obs, deterministic=True)
    all_actions.append(action[0]) # Assuming single environment, action is a list of [carer_idx]

    # Step the environment with the action
    # env.step() returns (next_obs_batch, reward_batch, terminated_batch, truncated_batch, info_batch)
    obs_batch, reward, terminated, truncated, info = env.step(action)
    current_obs = obs_batch[0] # Extract the single dictionary observation for the next iteration

    if terminated or truncated:
        break

# Convert collected actions and observations into a meaningful assignments DataFrame
# We now use 'all_observations' to retrieve the 'visit_id' that was processed at each step.
assigned_visit_ids = [o['visit_id'] for o in all_observations]
# Map action (carer index) to actual CarerID from carers DataFrame
assigned_carer_ids = [carers['CarerID'].iloc[carer_idx] for carer_idx in all_actions]

assignments_data = []
for i, visit_id in enumerate(assigned_visit_ids):
    carer_id = assigned_carer_ids[i]
    # Get start and end times for the visit using the original visits DataFrame
    visit_info = visits[visits['VisitID'] == visit_id].iloc[0]
    assignments_data.append({
        'VisitID': visit_id,
        'CarerID': carer_id,
        'StartTime': visit_info['TimeWindowStart'], # Using TimeWindowStart as a placeholder
        'EndTime': visit_info['TimeWindowEnd'] # Using TimeWindowEnd as a placeholder
    })

assignments = pd.DataFrame(assignments_data)

print("Generated assignments head:")
print(assignments.head())
print("Generated assignments shape:", assignments.shape)

# Now, re-evaluate with the generated assignments
metrics_df = evaluate_system(assignments, visits, carers)

print("\nUpdated metrics:")
print(metrics_df)

# Save the updated metrics
save_metrics(metrics_df, "/content/Project/results/ppo_metrics.csv")

KeyError: 0

In [35]:
print(type(env.assignments))
print(len(env.assignments))
print(env.assignments[:5])

<class 'list'>
0
[]


In [37]:
print(visits["Day"].unique())
print(carers["WorkingDaysSet"].iloc[0])

['tuesday' 'friday' 'thursday' 'wednesday' 'monday' 'sunday' 'saturday']
{'friday', 'wednesday', 'monday', 'thursday', 'saturday', 'tuesday'}


In [38]:
assignments_df = env.get_results()

print(assignments_df.shape)
print(assignments_df.head(10))

(0, 0)
Empty DataFrame
Columns: []
Index: []


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [41]:
import matplotlib.pyplot as plt

# Check if assignments_df is not empty and contains 'CarerID' before plotting
if 'CarerID' in assignments_df.columns and not assignments_df.empty:
    counts = assignments_df["CarerID"].value_counts().head(10)
    counts.plot(kind="bar")
    plt.title("Top Carer Utilization")
    plt.xlabel("Carer ID")
    plt.ylabel("Number of Assignments")
    plt.show()
else:
    print("Cannot plot Carer Utilization: assignments_df is empty or does not contain 'CarerID' column.")
    print(f"Current assignments_df columns: {assignments_df.columns.tolist()}")
    print(f"assignments_df is empty: {assignments_df.empty}")

Cannot plot Carer Utilization: assignments_df is empty or does not contain 'CarerID' column.
Current assignments_df columns: []
assignments_df is empty: True
